# 태스크 2: Strands를 사용하여 다중 에이전트 워크플로 구축

이 작업에서는 Strands Agents의 [’Agents as Tools’](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/multi-agent/agents-as-tools/) 패턴을 사용하여 단일 에이전트 시스템에서 정교한 다중 에이전트 아키텍처로 발전하게 됩니다. 오케스트레이터 아래 협력하여 포괄적인 재무 지침을 제공하는 전문 금융 에이전트를 구성하게 됩니다.

이 태스크는 전문 에이전트가 협업하여 단일 상담원의 능력을 넘어서는 복잡한 문제를 해결하는 엔터프라이즈급 에이전트 조정을 보여줍니다. 프로덕션 규모의 AI 시스템 구축에 필수적인 기술인 계층적 에이전트 설계, 도구 래핑 패턴, 지능형 요청 라우팅을 배우게 됩니다.

![아키텍처](./images/multi-agent_ko_kr.png)


### 시스템 아키텍처

멀티 에이전트 시스템은 세 가지 핵심 구성 요소로 구성됩니다.

#### 1. 예산 에이전트(태스크 1)
**개인 예산 책정, 지출 분석 및 재무 규율을 전문으로 합니다.**

| 도구 | 설명 | 예제 사용 사례 |
| --- | --- | --- |
| **calculate_budget_breakdown** | 모든 소득 수준에 대한 50/30/20 예산 계산 | 월 소득 6,000달러에 대한 예산 생성 |
| **analyze_spending_pattern** | 맞춤형 추천을 통한 지출 패턴 분석 | 소득 5,000달러 대비 800달러의 식사 비용 분석. |
| **calculator** | 재무 계산 및 수학 연산 | 예산의 20% 절감 목표를 계산 |
  
#### 2. 재무 분석 에이전트
**투자 조사, 포트폴리오 관리 및 시장 분석에 중점을 둡니다.**

| 도구 | 설명 | 예제 사용 사례 |
|------|-------------|------------------|
| **get_stock_analysis** | 실시간 주식 데이터 및 종합 분석 | Apple 주식 실적 및 지표를 분석해 주세요. |
| **create_diversified_portfolio** | 배분을 포함한 위험 기반 포트폴리오 권장 사항 | 10,000달러 예산의 중간 위험 포트폴리오 생성 |
| **compare_stock_performance** | 기간별 다중 주식 실적 비교 | 6개월 동안 Tesla, Apple, Google 비교 |

#### 3. 오케스트레이터 에이전트

**전문 에이전트를 조정하고 포괄적인 응답을 종합합니다.**

| 능력 | 설명 | 예제 사용 사례 |
|------------|-------------|------------------|
| **에이전트 라우팅** | 상담할 전문가를 지능적으로 결정 | 예산 관련 질문은 예산 담당자에게, 투자 문의는 재무 담당자에게 라우팅 |
| **다중 에이전트 조정** | 복잡한 쿼리에 대해 여러 에이전트의 통찰력을 결합 | 두 에이전트를 함께 사용하는 ‘예산 책정 및 투자 지원’ |
| **응답 종합** | 여러 에이전트의 결과로부터 일관된 응답 생성 | 예산 분석과 투자 권장 사항 결합 |
| **컨텍스트 관리** | 에이전트 상호작용 전반에 걸쳐 대화 흐름 유지 | 후속 권장 사항 작성 시 이전 조언 기억 |

In [ ]:
%%capture
# [환경 준비] 다중 에이전트 시스템에 필요한 라이브러리 설치.
# Task 2 는 Task 1 에서 만든 budget_agent.py 를 재사용하므로, Task 1 을 먼저 실행해 둬야 한다.
!pip install --force-reinstall -U -r requirements.txt --quiet --disable-pip-version-check

In [ ]:
# [임포트] 다중 에이전트 + 주식 분석에 필요한 모듈.
from strands import Agent, tool
from strands.models import BedrockModel
from strands.agent.conversation_manager import SummarizingConversationManager
import yfinance as yf                    # Yahoo Finance 데이터 조회 라이브러리(주가 이력)
from typing import List
from utils import create_guardrail
import boto3
import logging

# [리전 자동 감지] 하드코딩 대신 현재 자격 증명의 기본 리전을 읽는다.
region = boto3.Session().region_name

# [로깅 설정] 도구 실행 중 발생하는 예외를 추적한다.
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

### 태스크 2.1: 예산 에이전트를 도구로 활용

In [ ]:
# [도구로서의 에이전트 #1] Task 1 에서 만든 예산 에이전트를 '도구'로 감싼다.
# 핵심 패턴: 완성된 에이전트를 @tool 로 래핑하면, 상위 오케스트레이터가 호출할 수 있는 도구가 된다.
from budget_agent import FinancialReport, budget_agent  # Task 1 익스포트 파일에서 가져옴

@tool
def budget_agent_tool(query: str) -> FinancialReport:
    """Generate structured financial reports with budget analysis and recommendations."""
    try:
        # 내부적으로 Task 1 의 budget_agent 를 구조화 출력 모드로 실행한다.
        structured_response = budget_agent.structured_output(
            output_model=FinancialReport, prompt=query
        )
        return structured_response
    except Exception as e:
        # [점진적 성능 저하] 실패해도 오케스트레이터가 멈추지 않도록, 예외 대신 '기본 보고서'를 돌려준다.
        return FinancialReport(
            monthly_income=0.0,
            budget_categories=[],
            recommendations=[f"Error generating report: {str(e)}"],
            financial_health_score=1,
        )

### 태스크 2.2: 재무 분석 에이전트 생성

In [ ]:
# [시스템 프롬프트] 두 번째 전문 에이전트 = '재무 분석(투자 리서치)' 에이전트의 역할 정의.
# 예산 에이전트와 달리 주식･포트폴리오를 다루되, '구체적 투자 조언은 하지 않는다'는 경계를 둔다.
# 아래 프롬프트 문자열의 줄바꿈･빈 줄은 모델에 그대로 전달되므로 원본 형식을 유지한다.
FINANCIAL_ANALYSIS_PROMPT = """You are a specialized financial analysis agent focused on investment research and portfolio recommendations. Your role is to:

1. Research and analyze stock performance data
2. Create diversified investment portfolios
3. Provide data-driven investment recommendations

You do not provide specific investment advice but rather present analytical data to help users make informed decisions. Always include disclaimers about market risks and the importance of consulting financial advisors."""

In [ ]:
# [모델 설정] 재무 분석 에이전트용 모델. 여기서는 가드레일 없이 구성(테스트 단계).
bedrock_model = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    region_name=region,
    temperature=0.0,  # 재무 데이터는 일관성이 중요하므로 결정적으로
)

In [ ]:
# [도구] 특정 종목의 종합 분석. yfinance 로 실제 시장 데이터를 가져와 지표를 계산한다.
@tool
def get_stock_analysis(symbol: str) -> str:
    """Get comprehensive analysis for a specific stock symbol."""
    try:
        # [API 호출] yfinance 로 종목 객체를 만들고, 회사 정보와 1년치 일별 가격 이력을 받는다.
        stock = yf.Ticker(symbol)
        info = stock.info                      # 회사명·섹터 등 메타데이터(dict)
        hist = stock.history(period="1y")      # 1년 일별 OHLCV 데이터(pandas DataFrame)

        # [파싱·계산] DataFrame 에서 필요한 값을 뽑아 지표를 만든다.
        #   .iloc[-1] = 마지막 행(가장 최근) / .iloc[0] = 첫 행(1년 전)
        current_price = hist["Close"].iloc[-1]     # 현재가 = 종가 컬럼의 마지막 값
        year_high = hist["High"].max()             # 52주 최고가
        year_low = hist["Low"].min()               # 52주 최저가
        avg_volume = hist["Volume"].mean()         # 평균 거래량
        # 연초 대비 등락률(%) = (현재가 - 1년전가) / 1년전가 * 100
        price_change = (
            (current_price - hist["Close"].iloc[0]) / hist["Close"].iloc[0]
        ) * 100

        # info.get("키", "N/A") : 키가 없어도 KeyError 없이 기본값을 쓰는 안전한 접근
        return f"""
📊 Stock Analysis for {symbol.upper()}:
• Current Price: ${current_price:.2f}
• 52-Week High: ${year_high:.2f}
• 52-Week Low: ${year_low:.2f}
• Year-to-Date Change: {price_change:.2f}%
• Average Daily Volume: {avg_volume:,.0f} shares
• Company: {info.get("longName", "N/A")}
• Sector: {info.get("sector", "N/A")}
"""
    except Exception as e:
        # 잘못된 심볼·네트워크 오류 등을 문자열로 돌려줘 에이전트가 계속 동작하게 한다.
        return f"❌ Unable to retrieve data for {symbol}: {str(e)}"

In [ ]:
# [도구] 위험 성향별 분산 포트폴리오 추천.
@tool
def create_diversified_portfolio(risk_level: str, investment_amount: float) -> str:
    """Create a diversified portfolio based on risk level (conservative, moderate, aggressive) and investment amount."""
    # [데이터 구조] 위험 성향 -> {종목 리스트, 비중 리스트, 설명} 딕셔너리.
    # stocks 와 weights 는 '같은 순서로 짝지어진' 두 리스트다(아래 zip 에서 묶는다).
    portfolios = {
        "conservative": {
            "stocks": ["JNJ", "PG", "KO", "PEP", "WMT"],
            "weights": [0.25, 0.20, 0.20, 0.20, 0.15],
            "description": "Stable, dividend-paying blue-chip stocks with low volatility",
        },
        "moderate": {
            "stocks": ["AAPL", "MSFT", "JPM", "V", "DIS"],
            "weights": [0.25, 0.25, 0.20, 0.15, 0.15],
            "description": "Mix of established tech leaders and stable financial/consumer stocks",
        },
        "aggressive": {
            "stocks": ["TSLA", "NVDA", "META", "COIN", "PLTR"],
            "weights": [0.30, 0.25, 0.20, 0.15, 0.10],
            "description": "High-growth tech and emerging sector stocks with higher volatility",
        },
    }

    # [입력 검증] 지원하지 않는 위험 성향이면 즉시 안내 문자열 반환(딕셔너리 조회 전에 막는다).
    if risk_level.lower() not in portfolios:
        return "❌ Risk level must be: conservative, moderate, or aggressive"

    portfolio = portfolios[risk_level.lower()]

    # 아래 f-string 안의 빈 줄은 출력 포맷의 일부이므로 그대로 둔다.
    result = f"""
🎯 {risk_level.upper()} Portfolio Recommendation (${investment_amount:,.0f}):
{portfolio["description"]}

Portfolio Allocation:
"""

    # [반복문 + zip] 종목과 비중을 순서대로 짝지어 순회한다.
    #   zip(["AAPL","MSFT"], [0.25,0.25]) -> ("AAPL",0.25), ("MSFT",0.25)
    for stock, weight in zip(portfolio["stocks"], portfolio["weights"]):
        allocation = investment_amount * weight   # 이 종목에 배분할 금액 = 총액 x 비중
        result += f"• {stock}: {weight * 100:.0f}% (${allocation:,.0f})\n"

    result += "\n⚠️ Disclaimer: This is for educational purposes only. Consult a financial advisor before investing."
    return result

In [ ]:
# [도구] 여러 종목의 기간 수익률을 비교해 정렬해서 보여준다.
@tool
def compare_stock_performance(symbols: List[str], period: str = "1y") -> str:
    """Compare performance of multiple stocks over a specified period (1y, 6m, 3m, 1m)."""
    # [입력 검증] 너무 많으면 표가 지저분해지므로 5개로 제한.
    if len(symbols) > 5:
        return "❌ Please limit comparison to 5 stocks maximum"
    try:
        # [수집] 종목별 수익률을 딕셔너리에 모은다: {"AAPL": 12.3, "TSLA": -5.1, ...}
        performance_data = {}
        for symbol in symbols:                      # 각 종목을 하나씩 조회
            stock = yf.Ticker(symbol)
            hist = stock.history(period=period)
            if not hist.empty:                      # 데이터가 있을 때만 계산(빈 응답 방어)
                start_price = hist["Close"].iloc[0]   # 기간 시작가
                end_price = hist["Close"].iloc[-1]    # 기간 종료가
                performance = ((end_price - start_price) / start_price) * 100  # 수익률(%)
                performance_data[symbol] = performance

        result = f"📈 Stock Performance Comparison ({period}):\n"
        # [정렬] .items() 로 (종목, 수익률) 쌍을 만들고, key=lambda x: x[1] 로 '수익률' 기준 정렬.
        #        reverse=True -> 높은 수익률이 위로 오도록 내림차순.
        sorted_stocks = sorted(
            performance_data.items(), key=lambda x: x[1], reverse=True
        )
        # [반복문] 정렬된 순서대로 한 줄씩 출력. {performance:+.2f}% 의 '+' 는 양수에 부호를 붙인다(+12.30%).
        for stock, performance in sorted_stocks:
            result += f"• {stock}: {performance:+.2f}%\n"
        return result
    except Exception as e:
        return f"❌ Error comparing stocks: {str(e)}"

In [ ]:
# [에이전트 생성] 투자 분석 도구 3개를 갖춘 재무 분석 에이전트.
# LLM이 질문에 맞춰 세 도구 중 무엇을(때로는 여러 개를) 호출할지 스스로 결정한다.
financial_analysis_agent = Agent(
    model=bedrock_model,  # Task 2 앞부분에서 만든 모델 재사용
    system_prompt=FINANCIAL_ANALYSIS_PROMPT,
    tools=[get_stock_analysis, create_diversified_portfolio, compare_stock_performance],
)

In [ ]:
# [테스트] 포트폴리오 생성 + 종목 분석을 한 번에 요청.
# 모델이 create_diversified_portfolio 와 get_stock_analysis 를 순차로 호출할 것으로 기대된다.
response = financial_analysis_agent(
    "Create a moderate risk portfolio for $10,000 and analyze Apple stock"
)

In [ ]:
%%writefile financial_analysis_agent.py
# ============================================================================
# [파일 익스포트] %%writefile 은 이 셀 내용을 financial_analysis_agent.py 파일로 저장한다(코드 실행 아님).
# 재무 분석 에이전트(도구 3개 + 강화된 입력 검증)를 재사용 가능한 모듈로 저장한다.
# 노트북 앞 셀에서 만든 것을 한 파일로 합쳐, 다음 단계와 Task 3 에서 import 해 재사용한다.
# 아래 코드에는 노트북 버전보다 강화된 입력 검증･에러 처리가 포함될 수 있다.
# ============================================================================
# Export financial analysis agent to standalone Python file

import yfinance as yf
from strands import Agent, tool
from typing import List
from strands.models import BedrockModel
import logging
import boto3

# Get the current AWS region dynamically
region = boto3.Session().region_name

# Configure logging for error tracking and debugging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Financial Analysis Agent System Prompt
FINANCIAL_ANALYSIS_PROMPT = """You are a specialized financial analysis agent focused on investment research and portfolio recommendations. Your role is to:

1. Research and analyze stock performance data
2. Create diversified investment portfolios
3. Provide data-driven investment recommendations

You do not provide specific investment advice but rather present analytical data to help users make informed decisions. Always include disclaimers about market risks and the importance of consulting financial advisors."""

bedrock_model = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    region_name=region,
    temperature=0.0,  # Deterministic responses for financial advice
)


# Tool 1: Get Stock Analysis
@tool
def get_stock_analysis(symbol: str) -> str:
    """Get comprehensive analysis for a specific stock symbol."""
    try:
        # Fetch stock data
        stock = yf.Ticker(symbol)
        info = stock.info
        hist = stock.history(period="1y")
        
        if hist.empty:
            return f"❌ Error: No data found for symbol '{symbol.upper()}'."
        
        # Calculate key metrics
        current_price = hist["Close"].iloc[-1]
        year_high = hist["High"].max()
        year_low = hist["Low"].min()
        avg_volume = hist["Volume"].mean()
        price_change = ((current_price - hist["Close"].iloc[0]) / hist["Close"].iloc[0]) * 100

        return f"""
📊 Stock Analysis for {symbol.upper()}:
• Current Price: ${current_price:.2f}
• 52-Week High: ${year_high:.2f}
• 52-Week Low: ${year_low:.2f}
• Year-to-Date Change: {price_change:.2f}%
• Average Daily Volume: {avg_volume:,.0f} shares
• Company: {info.get("longName", "N/A")}
• Sector: {info.get("sector", "N/A")}
"""
    except Exception as e:
        logger.error(f"Error retrieving data for {symbol}: {e}")
        return f"❌ Error: Unable to retrieve stock data for '{symbol}'."


# Tool 2: Create Diversified Portfolio with comprehensive error handling
@tool
def create_diversified_portfolio(risk_level: str, investment_amount: float) -> str:
    """Create a diversified portfolio based on risk level (conservative, moderate, aggressive) and investment amount."""
    try:

        # Convert to float for calculations
        amount = float(investment_amount)
        
        # Input validation - negative check
        if amount < 0:
            logger.warning(f"Negative investment amount: {amount}")
            return "❌ Error: Investment amount cannot be negative. Please provide a positive amount."
        
        # Input validation - zero check
        if amount == 0:
            return "❌ Error: Investment amount cannot be zero. Please provide a positive amount to invest."
        
        # Input validation - minimum investment
        if amount < 100:
            return "❌ Error: Investment amount too small. Please provide at least $100 for portfolio diversification."
        
        # Input validation - maximum investment
        if amount > 100_000_000:
            logger.warning(f"Unusually high investment amount: {amount}")
            return "❌ Error: Investment amount seems unusually high. Please verify the amount (maximum $100,000,000)."
        
        portfolios = {
            "conservative": {
                "stocks": ["JNJ", "PG", "KO", "PEP", "WMT"],
                "weights": [0.25, 0.20, 0.20, 0.20, 0.15],
                "description": "Stable, dividend-paying blue-chip stocks with low volatility",
            },
            "moderate": {
                "stocks": ["AAPL", "MSFT", "JPM", "V", "DIS"],
                "weights": [0.25, 0.25, 0.20, 0.15, 0.15],
                "description": "Mix of established tech leaders and stable financial/consumer stocks",
            },
            "aggressive": {
                "stocks": ["TSLA", "NVDA", "META", "COIN", "PLTR"],
                "weights": [0.30, 0.25, 0.20, 0.15, 0.10],
                "description": "High-growth tech and emerging sector stocks with higher volatility",
            },
        }
        
        # Input validation - risk level check
        risk_level_lower = risk_level.strip().lower()
        if risk_level_lower not in portfolios:
            logger.warning(f"Invalid risk level provided: {risk_level}")
            return "❌ Error: Risk level must be 'conservative', 'moderate', or 'aggressive'. Please choose one of these options."
        
        portfolio = portfolios[risk_level_lower]
        
        result = f"""
🎯 {risk_level_lower.upper()} Portfolio Recommendation (${amount:,.0f}):
{portfolio["description"]}

Portfolio Allocation:
"""
        
        for stock, weight in zip(portfolio["stocks"], portfolio["weights"]):
            allocation = amount * weight
            result += f"• {stock}: {weight * 100:.0f}% (${allocation:,.0f})\n"
        
        result += "\n⚠️ Disclaimer: This is for educational purposes only. Consult a financial advisor before investing."
        return result
    
    except (ValueError, TypeError) as e:
        logger.error(f"Invalid input for create_diversified_portfolio: risk_level={risk_level}, amount={investment_amount}, error: {e}")
        return "❌ Error: Unable to create portfolio. Please check your inputs and try again."
    except Exception as e:
        logger.error(f"Unexpected error in create_diversified_portfolio: {e}")
        return "❌ Error: An unexpected error occurred while creating portfolio. Please try again."


# Tool 3: Compare Stock Performance with comprehensive error handling
@tool
def compare_stock_performance(symbols: List[str], period: str = "1y") -> str:
    """Compare performance of multiple stocks over a specified period (1y, 6m, 3m, 1m)."""
    try:
        # Input validation - type checking
        if not isinstance(symbols, list):
            logger.error(f"Invalid type for symbols: {type(symbols)}")
            return "❌ Error: Symbols must be provided as a list"
        
        # Input validation - empty list check
        if not symbols:
            return "❌ Error: Please provide at least one stock symbol to compare."
        
        # Input validation - list size check
        if len(symbols) > 5:
            return "❌ Error: Please limit comparison to 5 stocks maximum for better readability."
        
        # Input validation - period check
        valid_periods = ["1m", "3m", "6m", "1y", "2y", "5y"]
        if period not in valid_periods:
            logger.warning(f"Invalid period provided: {period}")
            return f"❌ Error: Invalid time period '{period}'. Please use one of: {', '.join(valid_periods)}"
        
        # Validate each symbol
        for symbol in symbols:
            if not isinstance(symbol, str):
                logger.error(f"Non-string symbol in list: {symbol}")
                return "❌ Error: All stock symbols must be text strings"
            if not symbol.strip():
                return "❌ Error: Empty stock symbol found. Please provide valid ticker symbols."
        
        performance_data = {}
        failed_symbols = []
        
        for symbol in symbols:
            symbol = symbol.strip().upper()
            try:
                stock = yf.Ticker(symbol)
                hist = stock.history(period=period)
                
                if not hist.empty and len(hist) >= 2:
                    start_price = hist["Close"].iloc[0]
                    end_price = hist["Close"].iloc[-1]
                    performance = ((end_price - start_price) / start_price) * 100
                    performance_data[symbol] = performance
                else:
                    failed_symbols.append(symbol)
            except Exception as e:
                logger.warning(f"Failed to fetch data for {symbol}: {e}")
                failed_symbols.append(symbol)
        
        # Check if we got any valid data
        if not performance_data:
            return f"❌ Error: Unable to retrieve data for any of the provided symbols. Please verify the ticker symbols are correct."
        
        result = f"📈 Stock Performance Comparison ({period}):\n"
        sorted_stocks = sorted(
            performance_data.items(), key=lambda x: x[1], reverse=True
        )
        
        for stock, performance in sorted_stocks:
            result += f"• {stock}: {performance:+.2f}%\n"
        
        # Add note about failed symbols if any
        if failed_symbols:
            result += f"\n⚠️ Note: Unable to retrieve data for: {', '.join(failed_symbols)}"
        
        return result
    
    except ValueError as e:
        logger.error(f"Value error in compare_stock_performance: {e}")
        return "❌ Error: Invalid input values. Please check your stock symbols and period."
    except Exception as e:
        logger.error(f"Unexpected error in compare_stock_performance: {e}")
        return "❌ Error: Unable to compare stock performance. Please check your internet connection and try again."


# Create the Financial Analysis Agent
financial_analysis_agent = Agent(
    model=bedrock_model,  # Using the same bedrock_model from Step 1
    system_prompt=FINANCIAL_ANALYSIS_PROMPT,
    tools=[get_stock_analysis, create_diversified_portfolio, compare_stock_performance],
    callback_handler=None,
)

if __name__ == "__main__":
    # Test the Financial Analysis Agent
    response = financial_analysis_agent(
        "Create a moderate risk portfolio for $10,000 and analyze Apple stock"
    )
    print(response)

In [ ]:
# [익스포트 검증] 저장한 financial_analysis_agent.py 를 실제로 실행해 본다.
# 이 파일은 다음 단계(오케스트레이터)와 Task 3 에서 import 로 재사용된다.
!python financial_analysis_agent.py 

### 태스크 2.3: 재무 분석 에이전트를 도구로 활용

In [ ]:
# [도구로서의 에이전트 #2] 재무 분석 에이전트를 도구로 감싼다.
# 방금 저장한 .py 에서 가져온다(노트북 셀 정의가 아니라 파일 버전을 쓴다).
from financial_analysis_agent import financial_analysis_agent

@tool
def financial_analysis_agent_tool(query: str) -> str:
    """Handle investment analysis queries including stock research, portfolio creation, and performance comparisons."""
    try:
        response = financial_analysis_agent(query)
        return str(response)   # 오케스트레이터에 넘기기 위해 응답을 문자열로 변환
    except Exception as e:
        return f"❌ Financial analysis error: {str(e)}"

### 태스크 2.4: 오케스트레이터 에이전트 생성

In [ ]:
# [오케스트레이터 프롬프트] 두 전문 에이전트를 '조율'하는 상위 에이전트의 역할 정의.
# 핵심은 라우팅 규칙: 어떤 질문을 어느 에이전트에게 위임할지 명시한다.
# 아래 프롬프트 문자열의 줄바꿈･빈 줄은 모델에 그대로 전달되므로 원본 형식을 유지한다.
ORCHESTRATOR_PROMPT = """You are a comprehensive financial advisor orchestrator that coordinates between specialized financial agents to provide complete financial guidance. 

Your specialized agents are:
1. **budget_agent**: Handles budgeting, spending analysis, savings recommendations, and expense tracking
2. **financial_analysis_agent_tool**: Handles investment analysis, stock research, portfolio creation, and performance comparisons

Guidelines for using your agents:
- Use **budget_agent** for questions about: budgets, spending habits, expense tracking, savings goals, debt management
- Use **financial_analysis_agent_tool** for questions about: stocks, investments, portfolios, market analysis, investment recommendations
- You can use both agents together for comprehensive financial planning
- Always provide a cohesive summary that combines insights from multiple agents when applicable
- Maintain a helpful, professional tone and include appropriate disclaimers about financial advice

When a user asks a question:
1. Determine which agent(s) are most appropriate
2. Call the relevant agent(s) with focused queries
3. Synthesize the responses into a coherent, comprehensive answer
4. Provide actionable next steps when possible"""

In [ ]:
# [가드레일] 오케스트레이터에도 콘텐츠 안전장치를 붙인다(사용자와 직접 대면하는 계층이므로).
guardrail_id, guardrail_arn = create_guardrail()

In [ ]:
# [모델 설정] 오케스트레이터용 모델. 가드레일을 연결한다.
bedrock_model = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    region_name=region,
    temperature=0.0,
    guardrail_id=guardrail_id,      # 가드레일 연결
    guardrail_version="DRAFT",
    guardrail_trace="enabled",
)

In [ ]:
# [대화 관리자] 오케스트레이터는 여러 에이전트 결과가 쌓여 컨텍스트가 길어지기 쉽다.
conversation_manager = SummarizingConversationManager(
    summary_ratio=0.3,           # 오래된 메시지의 30%를 요약으로 대체(예산 에이전트보다 보수적으로)
    preserve_recent_messages=5,  # 최근 5개는 원문 보존
)

In [ ]:
# [오케스트레이터 생성] 두 전문 에이전트를 '도구'로 보유한다.
# 사용자 -> 오케스트레이터 -> (예산 도구 / 분석 도구) 위임 -> 결과 종합, 의 다중 에이전트 구조 완성.
orchestrator_agent = Agent(
    model=bedrock_model,
    system_prompt=ORCHESTRATOR_PROMPT,
    tools=[budget_agent_tool, financial_analysis_agent_tool],  # 두 에이전트를 도구로 등록
    conversation_manager=conversation_manager,
)

In [ ]:
# [테스트] 두 에이전트가 모두 필요한 복합 질문.
# "테슬라·애플 비교(분석 에이전트) + 월 $4000 소득으로 $2000 투자 가능?(예산 에이전트)"
# -> 오케스트레이터가 두 도구를 모두 호출하고 결과를 하나로 합쳐야 한다.
response = orchestrator_agent("Compare Tesla and Apple stocks, and tell me if I can afford to invest $2000 with my $4000 monthly income.",)

In [ ]:
%%writefile main.py
# ============================================================================
# [파일 익스포트] %%writefile 은 이 셀 내용을 main.py 파일로 저장한다(코드 실행 아님).
# 예산･분석 두 에이전트를 조율하는 오케스트레이터 전체를 main.py 로 저장한다.
# 노트북 앞 셀에서 만든 것을 한 파일로 합쳐, 다음 단계와 Task 3 에서 import 해 재사용한다.
# 아래 코드에는 노트북 버전보다 강화된 입력 검증･에러 처리가 포함될 수 있다.
# ============================================================================
# Export complete multi-agent system to main.py

from strands import Agent, tool
from strands.models import BedrockModel
from strands.agent.conversation_manager import SummarizingConversationManager

from budget_agent import FinancialReport, budget_agent
from financial_analysis_agent import financial_analysis_agent

from utils import get_guardrail_id
import boto3

# Get the current AWS region dynamically
region = boto3.Session().region_name

ORCHESTRATOR_PROMPT = """You are a comprehensive financial advisor orchestrator that coordinates between specialized financial agents to provide complete financial guidance. 

Your specialized agents are:
1. **budget_agent**: Handles budgeting, spending analysis, savings recommendations, and expense tracking
2. **financial_analysis_agent_tool**: Handles investment analysis, stock research, portfolio creation, and performance comparisons

Guidelines for using your agents:
- Use **budget_agent** for questions about: budgets, spending habits, expense tracking, savings goals, debt management
- Use **financial_analysis_agent_tool** for questions about: stocks, investments, portfolios, market analysis, investment recommendations
- You can use both agents together for comprehensive financial planning
- Always provide a cohesive summary that combines insights from multiple agents when applicable
- Maintain a helpful, professional tone and include appropriate disclaimers about financial advice

When a user asks a question:
1. Determine which agent(s) are most appropriate
2. Call the relevant agent(s) with focused queries
3. Synthesize the responses into a coherent, comprehensive answer
4. Provide actionable next steps when possible"""

# Add conversation management to maintain context
conversation_manager = SummarizingConversationManager(
    summary_ratio=0.3,  # Summarize 30% of messages when context reduction is needed
    preserve_recent_messages=5,  # Always keep 5 most recent messages
)

# Continue with previous configurations
bedrock_model = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    region_name=region,
    temperature=0.0,  # Deterministic responses for financial advice
    guardrail_id=get_guardrail_id(),
    guardrail_version="DRAFT",
    guardrail_trace="enabled",
)


@tool
def budget_agent_tool(query: str) -> FinancialReport:
    """Generate structured financial reports with budget analysis and recommendations."""
    try:
        structured_response = budget_agent.structured_output(
            output_model=FinancialReport, prompt=query
        )
        return structured_response
    except Exception as e:
        # Return a default structured response on error
        return FinancialReport(
            monthly_income=0.0,
            budget_categories=[],
            recommendations=[f"Error generating report: {str(e)}"],
            financial_health_score=1,
        )


# Wrap Financial Analysis Agent as a Tool
@tool
def financial_analysis_agent_tool(query: str) -> str:
    """Handle investment analysis queries including stock research, portfolio creation, and performance comparisons."""
    try:
        response = financial_analysis_agent(query)
        return str(response)
    except Exception as e:
        return f"❌ Financial analysis error: {str(e)}"


orchestrator_agent = Agent(
    model=bedrock_model,
    system_prompt=ORCHESTRATOR_PROMPT,
    tools=[budget_agent_tool, financial_analysis_agent_tool],
    conversation_manager=conversation_manager,
)

if __name__ == "__main__":
    orchestrator_agent = Agent(
        model=bedrock_model,
        system_prompt=ORCHESTRATOR_PROMPT,
        tools=[budget_agent_tool, financial_analysis_agent_tool],
    )

    response = orchestrator_agent("I make $6000/month and want to start investing $500/month. Help me create a budget and suggest an investment portfolio.")

In [ ]:
# [최종 검증] 완성된 다중 에이전트 시스템(main.py)을 실행한다.
# Task 3 에서는 이 구조를 AgentCore Runtime 에 배포한다.
!python main.py 

**태스크 완료:** 예산 책정 및 투자 분석을 위해 전문 에이전트를 조정하는 계층적 다중 에이전트 시스템을 성공적으로 구축했습니다. 이제 시스템은 지능형 라우팅 및 조정을 통해 여러 전문 분야가 필요한 복잡한 쿼리를 처리할 수 있습니다.

### 다음 단계

이 노트북을 완료했습니다. 실습의 다음 부분으로 넘어가려면 실습 지침으로 돌아가서 **태스크 3**을 계속하십시오.